# Solutions · 17 · From messy laboratory data to a thermodynamic model

Worked solutions to the exercises of [notebook 17](../notebooks/17_from_messy_vle_data_to_a_model.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import io
from engthermo import activity, datasets, vapor, vle
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
raw = open(datasets.path("methanol_water_lab_vle.csv"), encoding="utf-8").read().splitlines()

def load_lab_vle(lines):
    fixed, log = [], []
    for line in lines:
        if line.startswith("#"):
            continue
        f = line.split(",")
        if len(f) == 7:
            f = f[:2] + [f[2] + "." + f[3]] + f[4:]
            log.append(f"decimal comma repaired in {f[0]}")
        fixed.append(",".join(f))
    d = pd.read_csv(io.StringIO("\n".join(fixed)))
    d["p_Pa"] = np.where(d.p_unit == "mmHg", d.p * 101325 / 760, d.p * 1e3)
    log.append(f"{(d.p_unit == 'mmHg').sum()} pressures converted from mmHg")
    kelvin = d["T"] > 200
    d["T_K"] = np.where(kelvin, d["T"], d["T"] + 273.15)
    log.append(f"{kelvin.sum()} temperature converted from K")
    dup = d.duplicated(subset=["x_methanol", "y_methanol", "T"])
    log.append(f"{dup.sum()} duplicate removed")
    d = d[~dup]
    missing = d.y_methanol.isna()
    log.append(f"{missing.sum()} run without vapour analysis dropped")
    d = d[~missing].sort_values("x_methanol").reset_index(drop=True)
    return d[["run", "x_methanol", "y_methanol", "T_K", "p_Pa"]], log

d, log = load_lab_vle(raw)
keep = d.run != "run-07"
comps = ["methanol", "water"]
x, y, T, p = d.x_methanol.to_numpy(), d.y_methanol.to_numpy(), d.T_K.to_numpy(), d.p_Pa.to_numpy()

## Exercise 1

Repeat the fit with the NRTL model. Do both models give the same limiting activity coefficients within the bootstrap uncertainty? Which parameter is least well determined, and which data would improve it?

In [3]:
rng = np.random.default_rng(1)
idx = np.nonzero(keep.to_numpy())[0]
out = {}
for model in ("wilson", "nrtl"):
    base = activity.fit(model, comps, x[idx], T[idx], p[idx], y1=y[idx])
    samples = []
    for _ in range(40):
        pick = rng.choice(idx, size=idx.size, replace=True)
        f = activity.fit(model, comps, x[pick], T[pick], p[pick], y1=y[pick])
        samples.append(activity.infinite_dilution(f["gamma"], 340.0))
    samples = np.array(samples)
    ginf = activity.infinite_dilution(base["gamma"], 340.0)
    print(f"{model:<7}: gamma_inf methanol in water {ginf[0]:.2f} +/- {samples[:, 0].std():.2f}, water in methanol {ginf[1]:.2f} +/- {samples[:, 1].std():.2f}")

wilson : gamma_inf methanol in water 2.31 +/- 0.05, water in methanol 1.68 +/- 0.03


nrtl   : gamma_inf methanol in water 2.29 +/- 0.05, water in methanol 1.64 +/- 0.05


The two models agree on both limiting activity coefficients within their bootstrap uncertainties. The
methanol-in-water limit is the less certain one: the data start at x = 0.02, and the dilute region is where
the models' shapes differ most. Two or three points below x = 0.02 would tighten it more than any
additional point in the middle of the range.

## Exercise 2

Suppose the barometer had been misread and the true pressure was 99.5 kPa throughout. Redo the analysis: how much do the fitted parameters change, and does the area test still pass? What does this say about recording the pressure?

In [4]:
for p_label, p_used in (("101.3 kPa as recorded", p), ("99.5 kPa (barometer corrected)", np.full_like(p, 99.5e3))):
    g1 = y * p_used / (x * vapor.antoine_psat("methanol", T)); g2 = (1 - y) * p_used / ((1 - x) * vapor.antoine_psat("water", T))
    ratio = np.log(g1 / g2)[keep]
    pos, neg = np.trapezoid(np.maximum(ratio, 0), x[keep]), -np.trapezoid(np.minimum(ratio, 0), x[keep])
    f = activity.fit("wilson", comps, x[keep], T[keep], p_used[keep], y1=y[keep])
    print(f"{p_label:<32}: Lambda12 = {f['params']['Lambda'][0, 1]:.3f}, Lambda21 = {f['params']['Lambda'][1, 0]:.3f}, area index {abs(pos - neg)/(pos + neg):.3f}")

101.3 kPa as recorded           : Lambda12 = 0.394, Lambda21 = 1.094, area index 0.046
99.5 kPa (barometer corrected)  : Lambda12 = 0.348, Lambda21 = 1.212, area index 0.046


A 2 % error in the pressure shifts both activity coefficients by 2 % - the area test, which depends on their
ratio, barely notices - but the fitted parameters move by several percent, and the change looks like a real
difference in the mixture. The barometer reading is as much part of a VLE measurement as the compositions,
and it belongs in the file.

## Exercise 3

**Implement it yourself:** turn steps 1-2 into a function `load_lab_vle(path)` returning a clean DataFrame in SI units and a list of the repairs made; test it on the file.

In [5]:
print(d.to_string(index=False)); print("\n".join(log))

   run  x_methanol  y_methanol    T_K     p_Pa
run-01        0.02       0.138 369.85 101300.0
run-02        0.05       0.270 365.55 101300.0
run-04        0.15       0.510 357.35 101325.0
run-05        0.20       0.579 354.95 101300.0
run-06        0.30       0.673 350.80 101300.0
run-07        0.40       0.622 348.15 101300.0
run-08        0.50       0.784 346.45 101325.0
run-09        0.60       0.833 344.65 101300.0
run-10        0.70       0.877 342.65 101300.0
run-11        0.80       0.924 341.05 101300.0
run-12        0.90       0.957 339.05 101300.0
run-13        0.95       0.979 338.55 101300.0
decimal comma repaired in run-11
2 pressures converted from mmHg
1 temperature converted from K
1 duplicate removed
1 run without vapour analysis dropped


The function reproduces every repair of the notebook and returns the clean table in SI units together with the
log - the reusable version of the workflow, ready for the next term's file.